In [ ]:
import os, math, json, random
from package import global_vars
from huggingface_hub import login
import matplotlib.pyplot as plt
import numpy as np
import pickle
from collections import Counter
from Tester import Tester

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor

In [ ]:


hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)
from package.items import Item
%matplotlib inline

In [ ]:
with open('train_lite.pkl', 'rb') as file:
    train=pickle.load(file)

with open('test_lite.pkl' , 'rb') as file:
    test = pickle.load(file)
print(train[0])
print(test[0])

In [ ]:
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
random.seed(42)
Tester.test(random_pricer)

In [ ]:
training_prices = [item.price for item in train]
training_avg = sum(training_prices)/len(training_prices)
def const_pricer(item):
    return training_avg

In [ ]:
Tester.test(const_pricer)

In [ ]:
for item in train:
    item.features = json.loads(item.details)
for item in test:
    item.features = json.loads(item.details)

In [ ]:
train[0].features.keys()

In [ ]:
feature_count = Counter()
for item in train:
    for f in item.features.keys():
        feature_count[f]+=1
feature_count.most_common(40)        

In [ ]:
def get_weight(item):
    weight_str = item.features.get('Item Weight')
    if weight_str:
        parts = weight_str.split(' ')
        amount = float(parts[0])
        unit = parts[1].lower()
        if unit=="pounds":
            return amount
        elif unit=="ounces":
            return amount / 16
        elif unit=="grams":
            return amount / 453.592
        elif unit=="milligrams":
            return amount / 453592
        elif unit=="kilograms":
            return amount / 0.453592
        elif unit=="hundredths" and parts[2].lower()=="pounds":
            return amount / 100
        else:
            print(weight_str)
    return None


In [ ]:
weights = [get_weight(t) for t in train]
weights = [w for w in weights if w]
average_weight = sum(weights)/len(weights)
average_weight
def get_weight_with_default(item):
    weight = get_weight(item)
    return weight or average_weight

In [ ]:
def get_rank(item):
    rank_dict = item.features.get("Best Sellers Rank")
    if rank_dict:
        ranks = rank_dict.values()
        return sum(ranks)/len(ranks)
    return None
ranks = [get_rank(t) for t in train]
ranks = [r for r in ranks if r]
average_rank = sum(ranks)/len(ranks)
average_rank    
def get_rank_with_default(item):
    rank = get_rank(item)
    return rank or average_rank

In [ ]:
def get_text_length(item):
    return len(item.test_prompt())

In [ ]:
TOP_ELECTRONICS_BRANDS = ["hp", "dell", "lenovo", "samsung", "asus", "sony", "canon", "apple", "intel"]
def is_top_electronics_brand(item):
    brand = item.features.get("Brand")
    return brand and brand.lower() in TOP_ELECTRONICS_BRANDS

In [ ]:
def get_features(item):
    return {
        "weight": get_weight_with_default(item),
        "rank": get_rank_with_default(item),
        "text_length": get_text_length(item),
        "is_top_electronics_brand": 1 if is_top_electronics_brand(item) else 0
    }
get_features(train[0])

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test[:250])

In [ ]:
np.random.seed(42)
features_column=['weight', 'rank', 'text_length', 'is_top_electronics_brand']

x_train = train_df[features_column]
y_train = train_df['price']

x_test = test_df[features_column]
y_test = test_df['price']
y_train

In [ ]:

model = LinearRegression()
model.fit(x_train, y_train)
for feature, coef in zip(features_column, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")
y_pred = model.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

In [ ]:
features = get_features(train[0])

In [ ]:
features

In [ ]:
def linear_reg_pricer(item):
    f = get_features(item)
    f_df = pd.DataFrame([f])
    return model.predict(f_df)[0]
Tester.test(linear_reg_pricer)    

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.test_prompt() for item in train]
np.random.seed(42)
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(documents)
regressor = LinearRegression()
regressor.fit(X, prices)
def bow_lr_pricer(item):
    x = vectorizer.transform([item.test_prompt()])
    return max(regressor.predict(x)[0],0)
Tester.test(bow_lr_pricer, test)

In [ ]:
np.random.seed(42)
processed_docs = [simple_preprocess(doc) for doc in documents]
w2v_model = Word2Vec(sentences=processed_docs, vector_size=400, window = 5, min_count=1, workers=8)
def doc_vector(doc):
    doc_words = simple_preprocess(doc)
    word_vectors = [w2v_model.wv[word] for word in doc_words if word in w2v_model.wv]
    return np.mean(word_vectors,axis=0) if word_vectors else np.zeros(word_vectors.vector_size)
x_w2v = np.array([doc_vector(doc) for doc in documents])

In [ ]:
processed_docs[0]

In [ ]:
w2v_lr = LinearRegression()
w2v_lr.fit(x_w2v, prices)

In [ ]:
def w2v_lr_pricer(item):
    doc = item.test_prompt()
    x_t = doc_vector(doc)
    return max(0,w2v_lr.predict([x_t])[0])
Tester.test(w2v_lr_pricer)

In [ ]:
# Support Vector Machines

np.random.seed(42)
svr_regressor = LinearSVR()

svr_regressor.fit(x_w2v, prices)
def svr_pricer(item):
    np.random.seed(42)
    doc = item.test_prompt()    
    return max(float(svr_regressor.predict([doc_vector(doc)])[0]),0)
Tester.test(svr_pricer)    

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=8)
rf_model.fit(x_w2v, prices)
def rf_pricer(item):
    doc = item.test_prompt()
    return max(rf_model.predict([doc_vector(doc)])[0],0)

In [ ]:
Tester.test(rf_pricer)